# 09 — 3D raw image workflow from spectrum-derived bins

This notebook demonstrates the workflow:

```text
calibrated mass spectrum CSV
→ automatic peak detection
→ tentative isotope assignment
→ mass-bin creation
→ load 3D raw FPD image layers
→ reconstruct mass-filtered 3D ion volumes
```

This workflow uses the **mass spectrum CSV** for mass calibration and peak assignment.
The **3D raw image files** are then filtered using the generated bins.

It does **not** use the depth-profile raw histogram workflow.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pymagsims import Spectrum, SIMSVolume
from pymagsims.isotopes import load_builtin_isotopes
from pymagsims.plotting import plot_ion_image_grid, plot_volume_slice

DATA = Path("../data")

## 1. Load calibrated mass spectrum

The spectrum CSV provides channel → mass calibration, intensity, metadata, and an optional ROI table.

In [ ]:
spectrum_path = DATA / "FPD_01_2604281458290.csv"

spec = Spectrum.from_main_analysis_file(spectrum_path)

display(spec.metadata)
display(spec.roi_table)
display(spec.data.head())

spec.plot(log_y=True);

## 2. Automatic peak detection and isotope assignment

Peak assignment here is tentative. SIMS peaks can also be molecular ions, hydrides, oxides, or clusters.

In [ ]:
isotopes = load_builtin_isotopes()

assignments = spec.assign_peaks(
    isotope_table=isotopes,
    tolerance=0.2,
    prominence=100,
    distance=5,
)

display(assignments.head(30))
print(f"Number of candidate assignments: {len(assignments)}")

## 3. Plot spectrum with identified peaks

In [ ]:
fig, ax, assignments = spec.plot_with_peaks(
    isotope_table=isotopes,
    tolerance=0.2,
    prominence=100,
    distance=5,
    log_y=True,
    annotate=True,
)

## 4. Create mass bins from identified peaks

The bins define mass ranges that are applied to every raw FPD image layer.
Adjust `width` depending on peak width and mass resolution.

In [ ]:
bins = spec.create_bins_from_assignments(
    assignments,
    width=0.3,
)

display(bins.head(20))

## 5. Select bins for 3D imaging

In [ ]:
# Keep a small subset for a fast first test.
selected_bins = bins.head(5).copy()

display(selected_bins)

# Alternative manual selection:
# selected_bins = bins[bins["label"].isin(["28Si", "69Ga", "16O"])].copy()

## 6. Load 3D raw image layer files

Each layer file has rows like:

```text
X ; Y ; Channel1 ; Channel2 ; Channel3 ; ...
```

Pixels with no detected events may be absent, so supply the image `shape` explicitly.

In [ ]:
from pathlib import Path
import re

DATA = Path("../data")
RAW_LAYER_DIR = DATA / "3d"

def natural_sort_key(path):
    return [
        int(text) if text.isdigit() else text.lower()
        for text in re.split(r"(\d+)", path.name)
    ]

paths = sorted(
    RAW_LAYER_DIR.glob("*Image_*.raw"),
    key=natural_sort_key,
)

print(f"Found {len(paths)} raw image layers")
for p in paths:
    print(p.name)

## 7. Build 3D ion volumes

Change `shape=(256, 256)` if your acquisition used another image size, e.g. `(512, 512)`.
The volume array shape is `(z, y, x)`.

In [ ]:
volume = SIMSVolume.from_fpd_raw_image_series(
    paths=paths,
    spectrum=spec,
    bins=selected_bins,
    include_total=True,
    shape=(256, 256),
)

print(volume.labels())
display(volume.metadata)

## 8. Plot one slice

In [ ]:
plot_volume_slice(
    volume,
    label="Total",
    z=0,
    log=True,
    cmap="viridis",
);

In [ ]:
label = [l for l in volume.labels() if l != "Total"][0]

plot_volume_slice(
    volume,
    label=label,
    z=0,
    log=True,
    cmap="magma",
);

## 9. Plot summed projections

Summed projections collapse the full stack along the depth axis.

In [ ]:
projection_images = {
    label: volume.sum_projection(label)
    for label in volume.labels()
}

plot_ion_image_grid(
    projection_images,
    log=True,
    ncols=3,
    cmaps=["gray", "viridis", "magma", "plasma", "cividis", "inferno", "turbo"],
);

## 10. Extract depth profiles from the 3D volume

In [ ]:
profiles = []

for label in volume.labels():
    profile = volume.depth_profile(label)
    profile = profile.rename(columns={label: "Intensity"})
    profile["Label"] = label
    profiles.append(profile)

depth_profiles = pd.concat(profiles, ignore_index=True)

display(depth_profiles.head())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

for label, group in depth_profiles.groupby("Label"):
    ax.plot(
        group["Slice"],
        group["Intensity"],
        marker="o",
        linewidth=1,
        label=label,
    )

ax.set_xlabel("Slice / layer")
ax.set_ylabel("Integrated counts")
ax.set_yscale("log")
ax.grid(True)
ax.legend()

fig.tight_layout()

## Summary

This workflow is appropriate when you have:

- one calibrated mass spectrum CSV
- one series of raw FPD image files

The spectrum provides mass calibration and bins.  
The raw image layers provide spatial information for 3D reconstruction.

## 11. Confirm 3D arrays for each bin / ion

The `SIMSVolume` object stores one 3D array per label:

```text
volume.volumes[label].shape = (z, y, x)
```

where:

- `z` = layer / scan number
- `y` = image row
- `x` = image column

In [ ]:
for label, arr in volume.volumes.items():
    print(f"{label}: shape={arr.shape}, total_counts={arr.sum()}")

## 12. Interactive Plotly layer slider

This viewer lets you scroll through the layers of one selected ion/bin.

Change `label` to inspect another ion channel.

In [ ]:
import plotly.graph_objects as go
import numpy as np

label = "Total"  # change to another label, e.g. volume.labels()[1]
arr = volume.get(label)

# log transform improves contrast for sparse ion images
arr_plot = np.log1p(arr)

z_count = arr_plot.shape[0]

fig = go.Figure()

# Add one heatmap trace per slice; only the first is visible initially.
for z in range(z_count):
    fig.add_trace(
        go.Heatmap(
            z=arr_plot[z],
            colorscale="Viridis",
            visible=(z == 0),
            colorbar=dict(title="log(1 + counts)"),
        )
    )

steps = []
for z in range(z_count):
    step = dict(
        method="update",
        args=[
            {"visible": [i == z for i in range(z_count)]},
            {"title": f"{label} — layer {z}"}
        ],
        label=str(z),
    )
    steps.append(step)

fig.update_layout(
    title=f"{label} — layer 0",
    xaxis_title="X pixel",
    yaxis_title="Y pixel",
    yaxis=dict(scaleanchor="x", autorange="reversed"),
    width=700,
    height=700,
    sliders=[
        dict(
            active=0,
            currentvalue={"prefix": "Layer: "},
            pad={"t": 50},
            steps=steps,
        )
    ],
)

fig

## 13. Interactive slider function

Use this helper to view any label more conveniently.

In [ ]:
def plot_volume_slider(volume, label="Total", log=True, colorscale="Viridis"):
    import plotly.graph_objects as go
    import numpy as np

    arr = volume.get(label)
    arr_plot = np.log1p(arr) if log else arr

    z_count = arr_plot.shape[0]

    fig = go.Figure()

    for z in range(z_count):
        fig.add_trace(
            go.Heatmap(
                z=arr_plot[z],
                colorscale=colorscale,
                visible=(z == 0),
                colorbar=dict(title="log(1 + counts)" if log else "counts"),
            )
        )

    steps = []
    for z in range(z_count):
        steps.append(
            dict(
                method="update",
                args=[
                    {"visible": [i == z for i in range(z_count)]},
                    {"title": f"{label} — layer {z}"}
                ],
                label=str(z),
            )
        )

    fig.update_layout(
        title=f"{label} — layer 0",
        xaxis_title="X pixel",
        yaxis_title="Y pixel",
        yaxis=dict(scaleanchor="x", autorange="reversed"),
        width=700,
        height=700,
        sliders=[
            dict(
                active=0,
                currentvalue={"prefix": "Layer: "},
                pad={"t": 50},
                steps=steps,
            )
        ],
    )

    return fig

In [ ]:
# Example usage:
plot_volume_slider(volume, label="Total", log=True)

In [ ]:
# View another ion/bin:
#plot_volume_slider(volume, label=volume.labels()[4], log=True, colorscale="Magma")